In [1]:
import os
from dotenv import load_dotenv
from mp_api.client import MPRester
import pandas as pd

load_dotenv()

MP_API_KEY = os.getenv("MP_API_KEY")

with MPRester(MP_API_KEY) as mpr:
    print("API Connection Sucessful")

API Connection Sucessful


In [5]:
# Define the fields we want to retrieve
fields = [
    "material_id",
    "formula_pretty",
    "structure",                # GNN Input
    "density",                  # MLP Input 1
    "nelements",                # MLP Input 2
    "symmetry",                 # MLP Input 3 (to get crystal_system)
    "band_gap",                 # Target 1
    "formation_energy_per_atom" # Target 2 (for stability screening)
]

In [7]:
with MPRester(MP_API_KEY) as mpr:
    # Query for semiconductor candidates
    # We filter for 'is_stable' to ensure high-quality training data
    docs = mpr.materials.summary.search(
        band_gap=(0.5, 5.0),      # Target range for hardware
        is_stable=True,           # Must be manufacture-able
        fields=fields
    )

Retrieving SummaryDoc documents:   0%|          | 0/13819 [00:00<?, ?it/s]

In [10]:
# Convert to a list of dictionaries
data_list = []
for doc in docs:
    data_list.append({
        "mp_id": str(doc.material_id),
        "formula": doc.formula_pretty,
        "structure": doc.structure,
        "density": doc.density,
        "nelements": doc.nelements,
        "crystal_system": str(doc.symmetry.crystal_system), # Extracted from symmetry
        "band_gap": doc.band_gap,
        "formation_energy": doc.formation_energy_per_atom
    })

In [11]:
df = pd.DataFrame(data_list)
df.shape

(13819, 8)

In [12]:
df.head(5)

,mp_id,formula,structure,density,nelements,crystal_system,band_gap,formation_energy
0,mp-11107,Ac2O3,"[[2.04774562 1.18226451 4.75519851] Ac, [-2.38...",9.109130,2,Trigonal,3.5226,-3.737668
1,mp-32800,Ac2S3,"[[ 2.1830569 1.2966013 19.29112704] Ac, [ 4...",6.535149,2,Tetragonal,2.2962,-2.493064
2,mp-1183115,AcAlO3,"[[0. 0. 0.] Ac, [1.92931694 1.92931694 1.92931...",8.728230,3,Cubic,4.1024,-3.690019
3,mp-27972,AcBr3,"[[4.0841257 2.35798973 3.54258841] Ac, [-2.43...",5.679086,2,Hexagonal,4.1033,-2.494519
4,mp-30274,AcBrO,"[[3.23277773 3.23277773 6.29799073] Ac, [1.077...",7.652290,3,Tetragonal,4.2410,-3.396186


In [14]:
int(df.isnull().sum().sum())

0

In [15]:
df.dtypes.value_counts()

str        3
float64    3
object     1
int64      1
Name: count, dtype: int64

In [16]:
from pathlib import Path
cwd = Path.cwd()

SAVE_PATH = cwd.parent / "data" / "materials_dataset_v1.pkl"

In [17]:
df.to_pickle(SAVE_PATH)

In [19]:
df[['formula', 'crystal_system', 'band_gap']].head()

,formula,crystal_system,band_gap
0,Ac2O3,Trigonal,3.5226
1,Ac2S3,Tetragonal,2.2962
2,AcAlO3,Cubic,4.1024
3,AcBr3,Hexagonal,4.1033
4,AcBrO,Tetragonal,4.2410
